# log-anomaly — walkthrough

Unsupervised anomaly detection over system logs, end to end on a small seeded
synthetic HDFS-like corpus. We:

1. run the one-command demo (`run_demo`) to get the real templating +
   event-count core and the ground-truth labels;
2. score the sessions with **two** label-free detectors — PCA reconstruction
   error and the robust **Mahalanobis** distance;
3. compare them honestly with **PR-AUC and ROC-AUC** (higher is better), using
   the labels only for *evaluation*, never for fitting.

Everything is pure numpy/pandas + stdlib, so it runs anywhere the package
imports.

## 1. Run the demo and rebuild the event-count matrix

`run_demo(0)` is deterministic. We re-derive the same event-count matrix `X`
and labels `y` it used, so the detectors below see exactly the demo's data.

In [ ]:
import numpy as np

import loganomaly as la
from loganomaly.demo import _anomalous_session, _normal_session, run_demo
from loganomaly.features import event_count_matrix
from loganomaly.templating import template_id

summary = run_demo(0)
keys = (
    "num_sessions",
    "num_templates",
    "num_anomalies_true",
    "precision",
    "recall",
    "f1",
)
print({k: summary[k] for k in keys})

In [ ]:
# Rebuild X and y deterministically from the same seed the demo uses.
rng = np.random.default_rng(0)
n_sessions, anomaly_rate = 300, 0.12
sessions, labels = {}, []
for _ in range(n_sessions):
    blk = f"blk_{rng.integers(-(1 << 62), 1 << 62)}"
    is_anom = bool(rng.random() < anomaly_rate)
    sessions[blk] = (
        _anomalous_session(rng, blk) if is_anom else _normal_session(rng, blk)
    )
    labels.append(is_anom)

table: dict[str, int] = {}
sess_ids = {
    b: [template_id(line, table) for line in lines] for b, lines in sessions.items()
}
n_templates = len(table)
X = event_count_matrix(sess_ids, n_templates)
y = np.array(labels, dtype=bool)
X.shape, int(y.sum())

## 2. Two unsupervised detectors

* **PCA reconstruction error** — distance from the dominant low-rank "normal"
  subspace.
* **Mahalanobis distance** — whitened distance from the centre, using a
  pseudo-inverse of the covariance so collinear / constant template columns do
  not blow it up.

Neither sees `y`.

In [ ]:
pca_scores = la.pca_reconstruction_error(X, k=3)
maha_scores = la.mahalanobis_scores(X)

# A third, even cheaper signal: template-rarity (IDF-weighted event counts).
idf = la.template_idf(sess_ids.values(), n_templates)
rarity = la.session_rarity(X, idf)
[round(float(s.max()), 3) for s in (pca_scores, maha_scores, rarity)]

## 3. Compare detectors by PR-AUC and ROC-AUC

The detectors emit a *continuous* score, so we don't need to pick one
threshold to compare them: we sweep all thresholds and integrate. ROC-AUC = 1.0
is perfect ranking; 0.5 is random. PR-AUC (average precision) is the more honest
summary on this imbalanced problem (~12% anomalies).

In [ ]:
import pandas as pd


def aucs(name, scores):
    fpr, tpr = la.roc_curve(y, scores)
    rec, prec = la.pr_curve(y, scores)
    return {
        "detector": name,
        "ROC-AUC": round(la.auc(fpr, tpr), 4),
        "PR-AUC": round(la.auc(rec, prec), 4),
    }


table_df = pd.DataFrame(
    [
        aucs("PCA reconstruction error", pca_scores),
        aucs("Mahalanobis distance", maha_scores),
        aucs("Template rarity (IDF)", rarity),
    ]
).set_index("detector")
table_df

## 4. Read it honestly

The AUCs rank the detectors *as rankers*, independent of any one cut. A higher
PR-AUC means that, at a matched recall, that detector flags fewer benign blocks.
But these are still statistical-rarity scores: a rare-but-benign block looks like
a rare-and-bad one, the labels can leak (they were assigned knowing the
failures), and the templates / normal subspace drift as the software evolves.
Treat a flag as a triage candidate, not a verdict — see `README.md` and
`USAGE.md` for the full caveats.